# AIS Trajectory Gap Reconstruction — Full Analysis Notebook

**Project:** Maritime Instruction Tuning for AIS Gap Reconstruction  
**Models evaluated:** LLM S0/C0 (3B), LLM S1/C1 (3B), LLM 1.5B (S0/C0), LSTM, GRU, BiLSTM  
**Gap sizes analyzed:** 5, 10, 30, 45 steps (= 50, 100, 300, 450 minutes)  

## Configuration Reference
| Config | SOG Tokenization | COG Tokenization | Model Size |
|---|---|---|---|
| S0/C0 | 5 coarse bins (STOP/LOW/MOD/HIGH/VHIGH) | 8-point compass (N/NE/…/NW) | 3B |
| S1/C1 | 10 domain-aware bins (<SOG_STOP>…<SOG_VFAST>) | 16-point compass (N/NNE/…/NNW) | 3B |
| 1.5B  | S0/C0 (same as above) | S0/C0 | 1.5B |

**Note:** The 1.5B model was only evaluated on gaps 30 and 45 using a different test set (300 samples/gap with 3 windows per trajectory). Direct per-sample comparison with 3B results is not valid; only aggregate-level cross-size comparisons are made.

**Note on test sets:** The S0/C0 and S1/C1 3B models use the same 100 test trajectories but trained separately (producing different baseline predictions). Per-sample comparisons between S0/C0 and S1/C1 use the same sample IDs.


## 0. Setup & Imports

In [1]:
import math
import os
import warnings
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import seaborn as sns

warnings.filterwarnings('ignore')
matplotlib.rcParams['figure.dpi'] = 120
plt.style.use('seaborn-v0_8-whitegrid')

# ── Paths ────────────────────────────────────────────────────────────────────
BASE    = Path('/home/av15650@ens.ad.etsmtl.ca/Downloads/Musit_instruction_tuning_3b_withOUTmlp+enhance sogcog')
BASE_1B5 = Path('/home/av15650@ens.ad.etsmtl.ca/Downloads/Musit_instruction_tuning (1.5b')
OUT     = BASE / 'report_outputs'
(OUT / 'tables').mkdir(parents=True, exist_ok=True)
(OUT / 'figures').mkdir(parents=True, exist_ok=True)

GAPS    = [5, 10, 30, 45]          # gap sizes (steps; 1 step = 10 min)
GAP_MIN = {5: 50, 10: 100, 30: 300, 45: 450}  # equivalent minutes

COLORS = {
    'LLM (S0/C0)': '#1f77b4',
    'LLM (S1/C1)': '#ff7f0e',
    'LSTM':        '#2ca02c',
    'GRU':         '#d62728',
    'BiLSTM':      '#9467bd',
    'LLM (1.5B)':  '#8c564b',
}
print('Setup complete.')

ModuleNotFoundError: No module named 'seaborn'

## 1. Metric Functions

In [ ]:
def haversine_km(lat1, lon1, lat2, lon2):
    """Great-circle distance between two WGS-84 points (km)."""
    R = 6371.0
    phi1, phi2 = math.radians(lat1), math.radians(lat2)
    dphi = math.radians(lat2 - lat1)
    dl   = math.radians(lon2 - lon1)
    a = math.sin(dphi/2)**2 + math.cos(phi1)*math.cos(phi2)*math.sin(dl/2)**2
    return R * 2 * math.asin(math.sqrt(max(0.0, a)))


def compute_per_sample_metrics(df_pred):
    """
    Compute per-sample ADE, FDE, DTW from a step-level DataFrame.

    Parameters
    ----------
    df_pred : DataFrame
        Must have columns: sample_id, trajectory_id, gap_steps, gap_step,
        pred_lat, pred_lon, true_lat, true_lon

    Returns
    -------
    DataFrame with one row per sample and columns: sample_id, trajectory_id,
    gap_steps, ADE_km, FDE_km, DTW_km.

    DTW note: for equal-length sequences the optimal warping path is the
    diagonal, so DTW = sum of pointwise haversine distances = ADE × n_steps.
    """
    records = []
    for sid, grp in df_pred.groupby('sample_id'):
        grp = grp.sort_values('gap_step')
        n = len(grp)
        if n == 0:
            continue
        dists = [haversine_km(r.true_lat, r.true_lon, r.pred_lat, r.pred_lon)
                 for r in grp.itertuples()]
        records.append({
            'sample_id':     sid,
            'trajectory_id': grp['trajectory_id'].iloc[0],
            'gap_steps':     int(grp['gap_steps'].iloc[0]),
            'ADE_km':        float(np.mean(dists)),
            'FDE_km':        float(dists[-1]),
            'DTW_km':        float(np.sum(dists)),
        })
    return pd.DataFrame(records)


def win_rate(llm_df, base_df, metric='ADE_km', label_base='LSTM'):
    """Per-gap win rate: fraction of samples where LLM metric < baseline metric."""
    records = []
    for g in GAPS:
        llm_g  = llm_df[llm_df['gap_steps'] == g][['sample_id', metric]].rename(columns={metric: 'llm'})
        base_g = base_df[base_df['gap_steps'] == g][['sample_id', metric]].rename(columns={metric: 'base'})
        merged = llm_g.merge(base_g, on='sample_id')
        if len(merged) == 0:
            continue
        wins  = (merged['llm'] < merged['base']).sum()
        total = len(merged)
        records.append({
            'Gap': g, 'Gap_min': GAP_MIN[g],
            f'wins vs {label_base}': int(wins),
            'Total': total,
            f'WinRate% vs {label_base}': round(100 * wins / total, 1),
        })
    return pd.DataFrame(records)


print('Metric functions defined.')

## 2. Load Data

In [ ]:
# ── S0/C0 configuration ─────────────────────────────────────────────────────
s0c0_llm = pd.read_csv(BASE / 'outputs/resultof_enhanced(sog_cog)/evaluation/test_predictions.csv')
s0c0_llm = s0c0_llm[s0c0_llm['gap_steps'].isin(GAPS)].copy()

s0c0_lstm_raw   = pd.read_csv(BASE / 'outputs/baselines/lstm_predictions.csv')
s0c0_gru_raw    = pd.read_csv(BASE / 'outputs/baselines/gru_predictions.csv')
s0c0_bilstm_raw = pd.read_csv(BASE / 'outputs/baselines/bilstm_predictions.csv')
for df in [s0c0_lstm_raw, s0c0_gru_raw, s0c0_bilstm_raw]:
    df.query('gap_steps in @GAPS', inplace=True)

s0c0_lstm   = compute_per_sample_metrics(s0c0_lstm_raw)
s0c0_gru    = compute_per_sample_metrics(s0c0_gru_raw)
s0c0_bilstm = compute_per_sample_metrics(s0c0_bilstm_raw)

# ── S1/C1 configuration ─────────────────────────────────────────────────────
s1c1_llm = pd.read_csv(BASE / 'outputs/S1_C2/evaluation_c1/test_predictions.csv')
s1c1_llm = s1c1_llm[s1c1_llm['gap_steps'].isin(GAPS)].copy()

s1c1_lstm_raw   = pd.read_csv(BASE / 'outputs/S1_C2/baselines/lstm_predictions.csv')
s1c1_gru_raw    = pd.read_csv(BASE / 'outputs/S1_C2/baselines/gru_predictions.csv')
s1c1_bilstm_raw = pd.read_csv(BASE / 'outputs/S1_C2/baselines/bilstm_predictions.csv')
for df in [s1c1_lstm_raw, s1c1_gru_raw, s1c1_bilstm_raw]:
    df.query('gap_steps in @GAPS', inplace=True)

s1c1_lstm   = compute_per_sample_metrics(s1c1_lstm_raw)
s1c1_gru    = compute_per_sample_metrics(s1c1_gru_raw)
s1c1_bilstm = compute_per_sample_metrics(s1c1_bilstm_raw)

# ── 1.5B model (gaps 30, 45 only; different test set) ────────────────────────
b15_llm  = pd.read_csv(BASE_1B5 / 'outputs/evaluation/test_predictions.csv')
b15_llm  = b15_llm[b15_llm['gap_steps'].isin([30, 45])].copy()

b15_lstm_raw   = pd.read_csv(BASE_1B5 / 'outputs/baselines/lstm_predictions.csv')
b15_gru_raw    = pd.read_csv(BASE_1B5 / 'outputs/baselines/gru_predictions.csv')
b15_bilstm_raw = pd.read_csv(BASE_1B5 / 'outputs/baselines/bilstm_predictions.csv')
for df in [b15_lstm_raw, b15_gru_raw, b15_bilstm_raw]:
    df.query('gap_steps in [30, 45]', inplace=True)

b15_lstm   = compute_per_sample_metrics(b15_lstm_raw)
b15_gru    = compute_per_sample_metrics(b15_gru_raw)
b15_bilstm = compute_per_sample_metrics(b15_bilstm_raw)

# ── Gap metadata (behavioral analysis) ──────────────────────────────────────
meta      = pd.read_csv(BASE / 'processed/gap_metadata.csv')
meta_test = meta[(meta['split'] == 'test') & (meta['gap_steps'].isin(GAPS))].copy()

# ── LLM per-step predictions (for trajectory plots) ──────────────────────────
llm_steps_path = BASE / 'outputs/S1_C2/evaluation/llm_per_step_predictions.csv'
llm_steps = pd.read_csv(llm_steps_path)
llm_steps = llm_steps[llm_steps['gap_steps'].isin(GAPS)].copy()

print(f'S0/C0 LLM: {len(s0c0_llm)} samples  |  S1/C1 LLM: {len(s1c1_llm)} samples')
print(f'1.5B LLM: {len(b15_llm)} samples  |  Gap metadata (test): {len(meta_test)} rows')
print(f'LLM per-step: {len(llm_steps)} rows')

## 3. Per-Gap Aggregate Metrics

In [ ]:
def aggregate_table(llm_df, lstm_df, gru_df, bilstm_df, cfg_label):
    rows = []
    for g in GAPS:
        for mname, mdf in [(f'LLM ({cfg_label})', llm_df),
                           ('LSTM', lstm_df), ('GRU', gru_df), ('BiLSTM', bilstm_df)]:
            sub = mdf[mdf['gap_steps'] == g]
            if len(sub) == 0:
                continue
            rows.append({'Gap': g, 'Gap_min': GAP_MIN[g], 'Model': mname, 'N': len(sub),
                         'ADE_km': round(sub['ADE_km'].mean(), 3),
                         'FDE_km': round(sub['FDE_km'].mean(), 3),
                         'DTW_km': round(sub['DTW_km'].mean(), 3)})
    return pd.DataFrame(rows)

tbl_s0c0 = aggregate_table(s0c0_llm, s0c0_lstm, s0c0_gru, s0c0_bilstm, 'S0/C0')
tbl_s1c1 = aggregate_table(s1c1_llm, s1c1_lstm, s1c1_gru, s1c1_bilstm, 'S1/C1')

print('=== S0/C0 Test Set ===')
display(tbl_s0c0.pivot(index='Model', columns='Gap', values='ADE_km'))
print('=== S1/C1 Test Set ===')
display(tbl_s1c1.pivot(index='Model', columns='Gap', values='ADE_km'))

In [ ]:
# Figure: per-gap ADE bar chart
def bar_chart_per_gap(tbl, metric, title, fname):
    fig, axes = plt.subplots(1, len(GAPS), figsize=(16, 5), sharey=False)
    for ax, g in zip(axes, GAPS):
        sub = tbl[tbl['Gap'] == g].copy()
        vals  = sub[metric].values
        models = sub['Model'].values
        cs = [COLORS.get(m, '#7f7f7f') for m in models]
        bars = ax.bar(range(len(models)), vals, color=cs, edgecolor='black', linewidth=0.5)
        best = int(np.argmin(vals))
        bars[best].set_edgecolor('gold'); bars[best].set_linewidth(2.5)
        ax.set_xticks(range(len(models)))
        ax.set_xticklabels(models, rotation=30, ha='right', fontsize=8)
        ax.set_title(f'Gap {g}\n({GAP_MIN[g]} min)', fontsize=10)
        ax.set_ylabel(metric if g == GAPS[0] else '')
        for bar, v in zip(bars, vals):
            ax.text(bar.get_x() + bar.get_width()/2, v + max(vals)*0.01,
                    f'{v:.2f}', ha='center', va='bottom', fontsize=7)
    fig.suptitle(title, fontsize=13, fontweight='bold')
    plt.tight_layout()
    plt.savefig(OUT / f'figures/{fname}', dpi=150, bbox_inches='tight')
    plt.show()

bar_chart_per_gap(tbl_s0c0, 'ADE_km', 'ADE per Gap — S0/C0 Test Set (gold = best)', 'ade_s0c0.png')
bar_chart_per_gap(tbl_s1c1, 'ADE_km', 'ADE per Gap — S1/C1 Test Set (gold = best)', 'ade_s1c1.png')
bar_chart_per_gap(tbl_s0c0, 'FDE_km', 'FDE per Gap — S0/C0 Test Set', 'fde_s0c0.png')
bar_chart_per_gap(tbl_s1c1, 'FDE_km', 'FDE per Gap — S1/C1 Test Set', 'fde_s1c1.png')

## 4. Per-Sample Win-Rate Analysis

In [ ]:
# Compute win rates for all metrics and baselines
wr_rows = []
for cfg, llm_df, lstm_df, gru_df, bi_df in [
    ('S0/C0', s0c0_llm, s0c0_lstm, s0c0_gru, s0c0_bilstm),
    ('S1/C1', s1c1_llm, s1c1_lstm, s1c1_gru, s1c1_bilstm),
]:
    for g in GAPS:
        llm_g = llm_df[llm_df['gap_steps'] == g]
        for base_name, base_df in [('LSTM', lstm_df), ('GRU', gru_df), ('BiLSTM', bi_df)]:
            for metric in ['ADE_km', 'FDE_km', 'DTW_km']:
                lv = llm_g[['sample_id', metric]].rename(columns={metric: 'llm'})
                bv = base_df[base_df['gap_steps'] == g][['sample_id', metric]].rename(columns={metric: 'base'})
                merged = lv.merge(bv, on='sample_id')
                if len(merged) == 0: continue
                wins = (merged['llm'] < merged['base']).sum()
                wr_rows.append({'Config': cfg, 'Gap': g, 'Gap_min': GAP_MIN[g],
                                'Baseline': base_name, 'Metric': metric,
                                'LLM_wins': int(wins), 'Total': len(merged),
                                'WinRate_%': round(100 * wins / len(merged), 1)})

wr_table = pd.DataFrame(wr_rows)
wr_table.to_csv(OUT / 'tables/combined_winrates.csv', index=False)
print('Win rates (ADE_km):')
display(wr_table[wr_table['Metric'] == 'ADE_km'].pivot_table(
    index=['Config', 'Baseline'], columns='Gap', values='WinRate_%'))

In [ ]:
# Win-rate heatmaps
def win_rate_heatmap(wr_table, cfg_label, metric, fname):
    baselines = ['LSTM', 'GRU', 'BiLSTM']
    sub = wr_table[(wr_table['Config'] == cfg_label) & (wr_table['Metric'] == metric)]
    data = np.zeros((len(GAPS), len(baselines)))
    for i, g in enumerate(GAPS):
        for j, b in enumerate(baselines):
            row = sub[(sub['Gap'] == g) & (sub['Baseline'] == b)]
            if len(row) > 0:
                data[i, j] = row.iloc[0]['WinRate_%']
    fig, ax = plt.subplots(figsize=(7, 4))
    im = ax.imshow(data, cmap='RdYlGn', vmin=0, vmax=100, aspect='auto')
    ax.set_xticks(range(len(baselines))); ax.set_xticklabels(baselines, fontsize=11)
    ax.set_yticks(range(len(GAPS))); ax.set_yticklabels([f'Gap {g}\n({GAP_MIN[g]} min)' for g in GAPS], fontsize=10)
    for i in range(len(GAPS)):
        for j in range(len(baselines)):
            ax.text(j, i, f'{data[i,j]:.0f}%', ha='center', va='center',
                    fontsize=12, fontweight='bold')
    plt.colorbar(im, ax=ax, label='LLM Win Rate (%)')
    ax.set_title(f'LLM ({cfg_label}) Win Rate — {metric}', fontsize=12)
    plt.tight_layout()
    plt.savefig(OUT / f'figures/{fname}', dpi=150, bbox_inches='tight')
    plt.show()

win_rate_heatmap(wr_table, 'S0/C0', 'ADE_km', 'winrate_s0c0_ade.png')
win_rate_heatmap(wr_table, 'S1/C1', 'ADE_km', 'winrate_s1c1_ade.png')
win_rate_heatmap(wr_table, 'S0/C0', 'FDE_km', 'winrate_s0c0_fde.png')
win_rate_heatmap(wr_table, 'S1/C1', 'FDE_km', 'winrate_s1c1_fde.png')

In [ ]:
# Win rate vs gap size line chart
fig, axes = plt.subplots(1, 2, figsize=(14, 5))
for ax, cfg in zip(axes, ['S0/C0', 'S1/C1']):
    sub = wr_table[(wr_table['Config'] == cfg) & (wr_table['Metric'] == 'ADE_km')]
    for base in ['LSTM', 'GRU', 'BiLSTM']:
        d = sub[sub['Baseline'] == base]
        ax.plot(d['Gap_min'], d['WinRate_%'], 'o-', label=f'vs {base}', linewidth=2, markersize=8)
    ax.axhline(50, color='gray', linestyle='--', linewidth=0.8, alpha=0.7)
    ax.set_xlabel('Gap Duration (minutes)'); ax.set_ylabel('LLM Win Rate (%)')
    ax.set_title(f'LLM ({cfg}) Win Rate (ADE)'); ax.set_ylim(0, 100); ax.legend(); ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.savefig(OUT / 'figures/winrate_vs_gap.png', dpi=150, bbox_inches='tight')
plt.show()

## 5. S0/C0 vs S1/C1 Configuration Comparison

In [ ]:
# Per-sample: when does S1/C1 beat S0/C0 and vice versa?
cfg_cmp_rows = []
for g in GAPS:
    for metric in ['ADE_km', 'FDE_km', 'DTW_km']:
        d0 = s0c0_llm[s0c0_llm['gap_steps'] == g][['sample_id', metric]].rename(columns={metric: 's0c0'})
        d1 = s1c1_llm[s1c1_llm['gap_steps'] == g][['sample_id', metric]].rename(columns={metric: 's1c1'})
        merged = d0.merge(d1, on='sample_id')
        if len(merged) == 0: continue
        s1c1_wins = (merged['s1c1'] < merged['s0c0']).sum()
        cfg_cmp_rows.append({'Gap': g, 'Gap_min': GAP_MIN[g], 'Metric': metric,
                             'S1C1_wins': int(s1c1_wins), 'Total': len(merged),
                             'S1C1_WinRate_%': round(100 * s1c1_wins / len(merged), 1)})

cfg_cmp = pd.DataFrame(cfg_cmp_rows)
cfg_cmp.to_csv(OUT / 'tables/cfg_comparison.csv', index=False)
display(cfg_cmp.pivot_table(index='Metric', columns='Gap', values='S1C1_WinRate_%'))

In [ ]:
# Scatter plot: S0/C0 vs S1/C1 ADE per sample
fig, axes = plt.subplots(1, len(GAPS), figsize=(16, 4))
for ax, g in zip(axes, GAPS):
    d0 = s0c0_llm[s0c0_llm['gap_steps'] == g][['sample_id', 'ADE_km']].rename(columns={'ADE_km': 's0c0'})
    d1 = s1c1_llm[s1c1_llm['gap_steps'] == g][['sample_id', 'ADE_km']].rename(columns={'ADE_km': 's1c1'})
    m  = d0.merge(d1, on='sample_id')
    ax.scatter(m['s0c0'], m['s1c1'], alpha=0.5, s=30, color='#1f77b4')
    lim = max(m['s0c0'].max(), m['s1c1'].max()) * 1.05
    ax.plot([0, lim], [0, lim], 'k--', linewidth=0.8, alpha=0.6)
    n_above = (m['s1c1'] < m['s0c0']).sum()  # S1/C1 better (below diagonal)
    ax.set_title(f'Gap {g} ({GAP_MIN[g]} min)\nS1/C1 better: {n_above}/{len(m)} ({100*n_above/len(m):.0f}%)')
    ax.set_xlabel('S0/C0 ADE (km)'); ax.set_ylabel('S1/C1 ADE (km)' if g == GAPS[0] else '')
fig.suptitle('S0/C0 vs S1/C1 — Per-Sample ADE (below diagonal = S1/C1 better)', fontsize=12)
plt.tight_layout()
plt.savefig(OUT / 'figures/scatter_cfg_comparison.png', dpi=150, bbox_inches='tight')
plt.show()

## 6. 1.5B Model Analysis

In [ ]:
# 1.5B aggregate metrics (different test set, 300 samples/gap, gaps 30 and 45 only)
b15_rows = []
for g in [30, 45]:
    for mname, mdf in [('LLM (1.5B)', b15_llm), ('LSTM', b15_lstm), ('GRU', b15_gru), ('BiLSTM', b15_bilstm)]:
        sub = mdf[mdf['gap_steps'] == g]
        if len(sub) == 0: continue
        b15_rows.append({'Gap': g, 'Gap_min': GAP_MIN[g], 'Model': mname, 'N': len(sub),
                         'ADE_km': round(sub['ADE_km'].mean(), 3),
                         'FDE_km': round(sub['FDE_km'].mean(), 3),
                         'DTW_km': round(sub['DTW_km'].mean(), 3)})
b15_tbl = pd.DataFrame(b15_rows)
b15_tbl.to_csv(OUT / 'tables/aggregate_1p5b.csv', index=False)
print('NOTE: 1.5B model uses a separate test set (300 samples/gap, different windows).')
display(b15_tbl.pivot(index='Model', columns='Gap', values='ADE_km'))

In [ ]:
# 1.5B win rates
b15_wr_rows = []
for g in [30, 45]:
    llm_g = b15_llm[b15_llm['gap_steps'] == g]
    for base_name, base_df in [('LSTM', b15_lstm), ('GRU', b15_gru), ('BiLSTM', b15_bilstm)]:
        for metric in ['ADE_km', 'FDE_km', 'DTW_km']:
            lv = llm_g[['sample_id', metric]].rename(columns={metric: 'llm'})
            bv = base_df[base_df['gap_steps'] == g][['sample_id', metric]].rename(columns={metric: 'base'})
            merged = lv.merge(bv, on='sample_id')
            if len(merged) == 0: continue
            wins = (merged['llm'] < merged['base']).sum()
            b15_wr_rows.append({'Gap': g, 'Gap_min': GAP_MIN[g], 'Baseline': base_name,
                                'Metric': metric, 'LLM_wins': int(wins), 'Total': len(merged),
                                'WinRate_%': round(100*wins/len(merged), 1)})
b15_wr = pd.DataFrame(b15_wr_rows)
b15_wr.to_csv(OUT / 'tables/winrates_1p5b.csv', index=False)
display(b15_wr[b15_wr['Metric'] == 'ADE_km'].pivot_table(index='Baseline', columns='Gap', values='WinRate_%'))

## 7. Behavioral Analysis

In [ ]:
# Join S1/C1 LLM per-sample metrics with gap metadata
llm_meta = s1c1_llm.merge(
    meta_test[['sample_id', 'gap_steps', 'observable_gap_duration_min',
               'mean_dt_inside_window_min', 'hidden_internal_duration_min',
               'max_dt_inside_window_min']],
    on=['sample_id', 'gap_steps'], how='inner')

bilstm_meta = s1c1_bilstm.merge(
    meta_test[['sample_id', 'gap_steps', 'observable_gap_duration_min',
               'mean_dt_inside_window_min']],
    on=['sample_id', 'gap_steps'], how='inner')

compare_meta = llm_meta.merge(
    bilstm_meta[['sample_id', 'gap_steps', 'ADE_km', 'FDE_km']].rename(
        columns={'ADE_km': 'bilstm_ADE', 'FDE_km': 'bilstm_FDE'}),
    on=['sample_id', 'gap_steps'])
compare_meta['llm_wins_ade'] = compare_meta['ADE_km'] < compare_meta['bilstm_ADE']

# Sampling regime: fast (<10 min between fixes) vs slow (>=10 min)
compare_meta['fast_sampling'] = compare_meta['mean_dt_inside_window_min'] < 10.0

print('LLM (S1/C1) vs BiLSTM win rate by gap and sampling regime (ADE):')
for g in GAPS:
    sub  = compare_meta[compare_meta['gap_steps'] == g]
    fast = sub[sub['fast_sampling']]
    slow = sub[~sub['fast_sampling']]
    wr_all  = sub['llm_wins_ade'].mean() * 100
    wr_fast = fast['llm_wins_ade'].mean() * 100 if len(fast) > 0 else float('nan')
    wr_slow = slow['llm_wins_ade'].mean() * 100 if len(slow) > 0 else float('nan')
    print(f'  Gap {g} ({GAP_MIN[g]} min):  all={wr_all:.1f}%  '
          f'fast (<10 min dt)={wr_fast:.1f}% (n={len(fast)})  '
          f'slow (>=10 min dt)={wr_slow:.1f}% (n={len(slow)})')

In [ ]:
# Observable gap duration vs LLM performance
fig, axes = plt.subplots(1, len(GAPS), figsize=(16, 4), sharey=True)
for ax, g in zip(axes, GAPS):
    sub = compare_meta[compare_meta['gap_steps'] == g]
    ax.scatter(sub['observable_gap_duration_min'], sub['ADE_km'],
               c=sub['llm_wins_ade'].map({True: '#2ca02c', False: '#d62728'}),
               alpha=0.6, s=30)
    ax.set_xlabel('Observable gap (min)'); ax.set_ylabel('LLM ADE (km)' if g == GAPS[0] else '')
    ax.set_title(f'Gap {g} ({GAP_MIN[g]} min)')
green = mpatches.Patch(color='#2ca02c', label='LLM wins vs BiLSTM')
red   = mpatches.Patch(color='#d62728', label='BiLSTM wins')
axes[-1].legend(handles=[green, red], fontsize=8)
fig.suptitle('LLM ADE vs Observable Gap Duration (S1/C1 vs BiLSTM)', fontsize=12)
plt.tight_layout()
plt.savefig(OUT / 'figures/behavioral_gap_duration.png', dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
# ADE distributions (boxplot)
def boxplot_ade(llm_df, lstm_df, gru_df, bilstm_df, cfg_label, fname):
    fig, axes = plt.subplots(1, len(GAPS), figsize=(18, 5))
    for ax, g in zip(axes, GAPS):
        data   = [df[df['gap_steps'] == g]['ADE_km'].values
                  for df in [llm_df, lstm_df, gru_df, bilstm_df]]
        labels = [f'LLM\n({cfg_label})', 'LSTM', 'GRU', 'BiLSTM']
        bp = ax.boxplot(data, labels=labels, patch_artist=True,
                        medianprops=dict(color='black', linewidth=2))
        for patch, c in zip(bp['boxes'],
                             [COLORS[f'LLM ({cfg_label})'], COLORS['LSTM'],
                              COLORS['GRU'], COLORS['BiLSTM']]):
            patch.set_facecolor(c); patch.set_alpha(0.7)
        ax.set_title(f'Gap {g} ({GAP_MIN[g]} min)')
        ax.set_ylabel('ADE (km)' if g == GAPS[0] else '')
        ax.tick_params(axis='x', labelsize=8)
    fig.suptitle(f'ADE Distribution — {cfg_label} Test Set', fontsize=12)
    plt.tight_layout()
    plt.savefig(OUT / f'figures/{fname}', dpi=150, bbox_inches='tight')
    plt.show()

boxplot_ade(s0c0_llm, s0c0_lstm, s0c0_gru, s0c0_bilstm, 'S0/C0', 'boxplot_s0c0.png')
boxplot_ade(s1c1_llm, s1c1_lstm, s1c1_gru, s1c1_bilstm, 'S1/C1', 'boxplot_s1c1.png')

## 8. Trajectory Visualizations

In [ ]:
def plot_trajectory(sid, gap, llm_steps_df, baseline_raw, bilstm_raw=None, title=''):
    llm_sub = llm_steps_df[(llm_steps_df['sample_id'] == sid) &
                            (llm_steps_df['gap_steps'] == gap)].sort_values('gap_step')
    bl_sub  = baseline_raw[(baseline_raw['sample_id'] == sid) &
                            (baseline_raw['gap_steps'] == gap)].sort_values('gap_step')
    if len(llm_sub) == 0 or len(bl_sub) == 0:
        print(f'No data for {sid} gap {gap}'); return

    true_lats = llm_sub['true_lat'].values
    true_lons = llm_sub['true_lon'].values
    llm_lats  = llm_sub['pred_lat'].values
    llm_lons  = llm_sub['pred_lon'].values
    bl_lats   = bl_sub['pred_lat'].values
    bl_lons   = bl_sub['pred_lon'].values

    fig, ax = plt.subplots(figsize=(8, 6))
    ax.plot(true_lons, true_lats, 'ko-', lw=2, ms=6, label='Ground Truth', zorder=5)
    ax.plot(llm_lons, llm_lats,   'b^--', lw=1.5, ms=5, label='LLM (S1/C1)', zorder=4)
    ax.plot(bl_lons, bl_lats,     'rv--', lw=1.5, ms=5, label='LSTM', zorder=3)
    if bilstm_raw is not None:
        bi_sub = bilstm_raw[(bilstm_raw['sample_id'] == sid) &
                             (bilstm_raw['gap_steps'] == gap)].sort_values('gap_step')
        if len(bi_sub) > 0:
            ax.plot(bi_sub['pred_lon'].values, bi_sub['pred_lat'].values,
                    'ms--', lw=1.5, ms=5, label='BiLSTM', zorder=3)
    ax.scatter([true_lons[0]],  [true_lats[0]],  c='lime', s=120, zorder=7, marker='*', label='Start')
    ax.scatter([true_lons[-1]], [true_lats[-1]], c='red',  s=120, zorder=7, marker='*', label='End')
    ax.set_xlabel('Longitude'); ax.set_ylabel('Latitude')
    ax.set_title(f'{title}  |  {sid} gap {gap} ({GAP_MIN[gap]} min)')
    ax.legend(fontsize=8, loc='best'); plt.tight_layout()
    plt.savefig(OUT / f'figures/traj_{sid}_gap{gap}.png', dpi=150, bbox_inches='tight')
    plt.show()

# Plot representative samples for each gap
for g in GAPS:
    gap_samples = llm_steps[llm_steps['gap_steps'] == g]['sample_id'].unique()
    for sid in gap_samples[:2]:
        plot_trajectory(sid, g, llm_steps, s1c1_lstm_raw, s1c1_bilstm_raw)

## 9. Token Accuracy Summary

In [ ]:
token_rows = []
for cfg, llm_df in [('S0/C0', s0c0_llm), ('S1/C1', s1c1_llm)]:
    for g in GAPS:
        sub = llm_df[llm_df['gap_steps'] == g]
        if 'exact_token_accuracy' in sub.columns and len(sub) > 0:
            token_rows.append({'Config': cfg, 'Gap': g, 'Gap_min': GAP_MIN[g],
                               'Token_Acc': round(sub['exact_token_accuracy'].mean(), 3),
                               'Seq_Exact_Match': round(sub['sequence_exact_match'].mean(), 3)})
tok_df = pd.DataFrame(token_rows)
display(tok_df.pivot_table(index='Config', columns='Gap', values='Token_Acc'))
display(tok_df.pivot_table(index='Config', columns='Gap', values='Seq_Exact_Match'))

## 10. Summary Tables Export
All tables are saved to `report_outputs/tables/`. All figures to `report_outputs/figures/`.

In [ ]:
tbl_s0c0.to_csv(OUT / 'tables/aggregate_s0c0.csv', index=False)
tbl_s1c1.to_csv(OUT / 'tables/aggregate_s1c1.csv', index=False)
wr_table.to_csv(OUT / 'tables/combined_winrates.csv', index=False)
cfg_cmp.to_csv(OUT / 'tables/cfg_comparison.csv', index=False)
compare_meta.to_csv(OUT / 'tables/behavioral_llm_vs_bilstm.csv', index=False)
print(f'All outputs saved to {OUT}')